In [7]:
import matplotlib.pyplot as plt
import parselib
import sys
import seaborn as sns
import numpy as np
import matplotlib as mpl
import plotconfig
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [ ]:
# CHANGE THIS TO GENERATE FIGURE 10 OR FIGURE 17
figure = 10

if figure not in [10,17]:
    print("Invalid figure no.")
    sys.exit()

In [ ]:
bbr_version = "bbr3" if figure == 10 else "bbr"

args = {
    "chaos": "on_1",
    "timestamp": ["2025", "2026052", "2026052", "2026052"],
    "cca": ["cubic", bbr_version, f"{bbr_version}-patched-new-new"],
    "test_cca": "yes",
    "kernel": ["kernel6-1", "zkernel6-13-BBRv3", "zkernel6-13-BBRv3-patched_new_new"],

    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [100],
    "delay_rtt": [10,20,30],
    "deadline_run": [1000000, 10000000, 20000000],
}

if bbr_version == "bbr":
    args["kernel"] = ["kernel6-1", "kernel6-1-patched_new_new"]

metric = "bits_per_second"

baselogpath = "../data"

In [9]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 10770
end 10770
bytes 10770
bits_per_second 10770
mbps_timeseries 10770
rttms_timeseries 10770
retransmits 10770
timestamp 10770
iteration 10770
cpu_host_total 10770
cpu_host_user 10770
cpu_host_system 10770
cpu_remote_total 10770
chaos 10770
deadline_run 10770
deadline_period 10770
os 10770
bdp 10770
setup 10770
cca 10770
cpus 10770
kernel 10770
mode 10770
loss 10770
rate 10770
delay_rtt 10770
buffer_size_bytes 10770
parallel 10770
socket_buffer 10770
app_buffer 10770
n 10770
sysctl_cmd 10770
vm 10770
bandwidth_delay_product 10770
loss_mode 10770
vms 10770
pacing 10770
hyperthreading 10770
tso 10770
qdisc 10770
hpet 10770
tsc 10770
hostq 10770
loadperc 10770
deadline_period_factor 10770
random_loss_rate 10770
gemodel_q 10770
original_cca 10770
test_cca 10770
default_qdisc 10770
json 10770


In [10]:
df["slice_perc"] = round((df["deadline_run"]/df["deadline_period"])*100)
df["cca_generic"] = df["cca"].apply(lambda x: x.replace("-patched", ""))
df["cca_generic"] = df["cca_generic"].apply(lambda x: x.replace("-new", ""))
df["patch_or_not"] = df["cca"].apply(lambda x: "apatched" if "-patched" in x else "original")
df["mbps"] = df["bits_per_second"]/1000000

replace_label = {
    "bbr": "BBRv1",
    "bbr2": "BBRv2",
    "bbr3": "BBRv3",
    "cubic": "Cubic"
}

In [11]:
plotconfig.configure_conext()
width = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)
height = width*(1/3)
FIG_SIZE = (width, height)

def strip(df, savefig = False):    
    if savefig:
        mpl.use('agg')
    
    paletti = {"apatched": plotconfig.COLORS[1], "original": "gainsboro"}
    fig, axs = plt.subplots(1,3,figsize=FIG_SIZE, sharey=True,constrained_layout=True)

    df_ = df[df["slice_perc"] % 5 == 0].sort_values(by=['patch_or_not'])
    cca = bbr_version


    for it, runti in enumerate(sorted(df["deadline_run"].unique())):
        df = df_[df_["deadline_run"] == runti]   

        data_sorted=df[df["cca_generic"] == cca].sort_values(by=['patch_or_not','slice_perc', 'delay_rtt'])
        extra_legend_patches = []
        
        for rtt_ in [10,20,30]:
            if rtt_ == 10:
                marker = "o"
            elif rtt_ == 20:
                marker = "v"
            elif rtt_ == 30:
                marker = "s"
            elif rtt_ == 40:
                marker = "P"
            extra_legend_patches.append(Line2D([0], [0], linestyle='none', mfc=paletti["apatched"], markersize=3,mec=paletti["apatched"], marker=marker, label=f'{int(rtt_)}ms'))
            data_ = data_sorted[["slice_perc", "mbps","patch_or_not", "delay_rtt"]]
            grouped_data = data_[data_["delay_rtt"] == rtt_].groupby(["slice_perc", "patch_or_not", "delay_rtt"],as_index=False).median()
            sns.stripplot(ax=axs[it], data=grouped_data, x="slice_perc", y="mbps", hue="patch_or_not", dodge=True, jitter=False, alpha=0.6, palette= paletti, zorder=0, legend=True if rtt_ == 10 else False, marker = marker, size=3,linewidth=0.1)            

        axs[it].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11,12], ["10","","20","","30","","40","","50","","60","","70"], fontsize=plotconfig.FONT_SIZE-3)
        axs[it].tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-3)
        axs[it].vlines([0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5,11.5], ymin=-5, ymax = int(args["rate"][0])+5, color="gainsboro", linewidth=0.5)
    

        df_cubic = df[df["cca"] == "cubic"]
        df_cubic = df_cubic[df_cubic["kernel"] == "kernel6-1"]
        df_cubic = df_cubic.sort_values(by=['slice_perc', "delay_rtt"])
        df_cubic = df_cubic[["slice_perc", "mbps","patch_or_not", "delay_rtt"]].groupby(["slice_perc", "patch_or_not", "delay_rtt"],as_index=False).median()
        for rtt_ in df_cubic["delay_rtt"].unique():
            if rtt_ == 10:
                marker = "o"
            elif rtt_ == 20:
                marker = "v"
            elif rtt_ == 30:
                marker = "s"
            elif rtt_ == 40:
                marker = "P"
            sns.stripplot(ax=axs[it], data=df_cubic[df_cubic["delay_rtt"] == rtt_], x="slice_perc", y="mbps", dodge=True, hue="patch_or_not", legend=False, jitter=True, alpha=0.9, palette=paletti, zorder=0, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)

        axs[it].set_title(f"Timeslice {int(runti/1000000)}ms",fontsize=plotconfig.FONT_SIZE-3,pad=3)
        if runti == 1000000:
            axs[it].set_xlabel("")
            axs[it].legend([])
        elif runti == 10000000:
            axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-3)
            axs[it].legend([])
        elif runti == 20000000:
            axs[it].set_xlabel("")
            han = [
                Line2D([0], [0], linestyle='none', mfc=paletti["apatched"], mec="black", mew=0.1, marker="o", markersize=3, alpha=0.6, label=replace_label[cca]),
                Line2D([0], [0], linestyle='none', mfc=paletti["apatched"], mec="black", mew=0.6, marker="o", markersize=1.8, alpha=0.9, label='Cubic'),
            ]
            leg1 = axs[it].legend(handles = han,loc="lower right", bbox_to_anchor=(1.3, 0.255), handlelength=1, framealpha=0.4,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3)
            axs[it].add_artist(leg1)

            h, l = axs[it].get_legend_handles_labels()
            for enu,hii in enumerate(h):
                hii.set_alpha(1)
                l[enu] = "original" if "original" in l[enu] else "patch"
            leg2 = axs[it].legend(handles=h, labels= l, loc="lower right", handlelength=0.9, bbox_to_anchor=(1.3,0.4), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)
            axs[it].add_artist(leg2)

            axs[it].legend(handles=extra_legend_patches, title="RTT", loc="lower right", handlelength=1.8, bbox_to_anchor=(1.3, 0), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)

    axs[0].set_ylabel("Avg. throughput [Mbps]",fontsize=plotconfig.FONT_SIZE-3)
    axs[0].set_ylim(-float(args["rate"][0])*0.05,int(args["rate"][0]))
    
    if savefig:
        fig.savefig(f"figures/figure_{'10' if bbr_version=="bbr3" else '17'}.pdf", format="pdf")
    else:
        plt.show()

<>:62: SyntaxWarning: invalid escape sequence '\%'
<>:62: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_242102/1082823579.py:62: SyntaxWarning: invalid escape sequence '\%'
  axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-3)


In [12]:
strip(df, True)